# Glass Classification

**Tabular Multi-Class Classification** — Classify glass types from chemical composition.

Models: CatBoost · LightGBM · XGBoost  
Baselines: LazyPredict · FLAML AutoML  
Dataset: UCI Glass (214 rows × 10 columns)  
Target: `Type` (6 classes)

## 2 · Project Overview

We classify **glass samples** into 6 types based on their oxide content (RI, Na, Mg, Al, Si, K, Ca, Ba, Fe).

With only 214 rows and 6 classes, this is a small-data multi-class challenge where simple models may compete with complex ones.

## 3 · Learning Objectives

1. Handle a small multi-class dataset (214 rows, 6 classes).
2. Understand feature importance in material science.
3. Compare boosting models on small tabular data.
4. Use LazyPredict and FLAML for rapid benchmarking.
5. Evaluate with accuracy, weighted F1, and per-class metrics.

## 4 · Problem Statement

Given the refractive index and 8 oxide measurements of a glass sample, predict its **Type** (1, 2, 3, 5, 6, or 7).

## 5 · Why This Project Matters

- Forensic glass analysis is used in criminal investigations.
- Material science relies on chemical composition for classification.
- Small datasets test model robustness and overfitting.
- Multi-class with missing class (no Type 4) is realistic.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 214 |
| **Columns** | 10 |
| **Features** | RI, Na, Mg, Al, Si, K, Ca, Ba, Fe |
| **Target** | Type (6 classes: 1, 2, 3, 5, 6, 7) |
| **Missing values** | None |
| **Source** | UCI ML Repository / local CSV |

## 7 · Dataset Source and License Notes

- **Source**: UCI Machine Learning Repository.
- **Creator**: B. German, Central Research Establishment, Home Office Forensic Science Service.
- **License**: Public / educational use.
- **Note**: Type 4 (non-float processed vehicle windows) is absent from the dataset.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, re, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "Type"
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data", "glass.csv")
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Data path: {DATA_PATH}")
print(f"Save dir : {SAVE_DIR}")

## 11 · Dataset Download or Loading

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

## 12 · Data Validation Checks

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget distribution:\n{df[TARGET].value_counts().sort_index()}")
print(f"\nClasses: {sorted(df[TARGET].unique())}")
print(f"Note: Type 4 is missing from the dataset.")
print(f"Shape: {df.shape}")

## 13 · Exploratory Data Analysis

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
features = [c for c in df.columns if c != TARGET]
for i, col in enumerate(features):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=25, ax=ax, color="steelblue", edgecolor="black", alpha=0.8)
    ax.set_title(col, fontsize=10)
plt.suptitle("Feature Distributions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = df.corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, square=True)
ax.set_title("Feature Correlation")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
df[TARGET].value_counts().sort_index().plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Glass Type Distribution")
ax.set_xlabel("Type")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

for t in sorted(df[TARGET].unique()):
    n = (df[TARGET] == t).sum()
    print(f"  Type {t}: {n} ({100*n/len(df):.1f}%)")

## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split. With 214 rows, cross-validation would be more robust but we keep it simple.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET].values

# Encode to 0-based for XGBoost
le = LabelEncoder()
y = le.fit_transform(y)
n_classes = len(le.classes_)

X.columns = [re.sub(r"[^A-Za-z0-9_]", "_", c) for c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {n_classes}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **All features numeric**: No encoding needed.
- **Scaling**: Not needed for tree models.
- **Target**: LabelEncoder to 0-based for XGBoost compatibility.

## 17 · Feature Engineering

The dataset is already well-designed with meaningful chemical features. No additional engineering needed for this small dataset.

## 18 · Baseline Model

In [ ]:
baseline = DecisionTreeClassifier(max_depth=5, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="weighted")

print("=" * 50)
print("BASELINE: Decision Tree (max_depth=5)")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print(f"\n{classification_report(y_test, y_pred_bl)}")

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

In [ ]:
from flaml import AutoML

try:
    flaml_automl = AutoML()
    flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                     metric="f1", verbose=0, seed=SEED)
    y_pred_flaml = flaml_automl.predict(X_test)
    acc_flaml = accuracy_score(y_test, y_pred_flaml)
    f1_flaml = f1_score(y_test, y_pred_flaml, average="weighted")
    print(f"FLAML best: {flaml_automl.best_estimator}")
    print(f"Accuracy: {acc_flaml:.4f}, F1: {f1_flaml:.4f}")
except Exception as e:
    print(f"FLAML failed: {e}")
    print("Using baseline predictions as FLAML fallback.")
    y_pred_flaml = y_pred_bl

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    cb = CatBoostClassifier(iterations=200, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    lg = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                            device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
    lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
           callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    xgb_model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                              device="cuda", tree_method="hist", verbosity=0,
                              eval_metric="mlogloss", n_jobs=-1, random_state=SEED)
    xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost: {e}")
gpu_cleanup()

## 22 · Top 3 Model Selection

In [ ]:
# Add baseline and FLAML
results["Baseline"] = y_pred_bl
results["FLAML"] = y_pred_flaml

model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], colorbar=False)
    f1 = f1_score(y_test, yp, average='weighted')
    axes[i].set_title(f"{name}\nF1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
    print(f"\n  {name}:")
    print(f"    Accuracy: {accuracy_score(y_t, yp):.4f}")
    print(f"    F1      : {f1_score(y_t, yp, average='weighted'):.4f}")
    print(f"    Precision: {precision_score(y_t, yp, average='weighted'):.4f}")
    print(f"    Recall  : {recall_score(y_t, yp, average='weighted'):.4f}")

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]

y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
misclassed = (y_t != best_pred)
n_wrong = misclassed.sum()
print(f"Best model: {best_name}")
print(f"Misclassified: {n_wrong} / {len(y_t)} ({100*n_wrong/len(y_t):.1f}%)")

if n_wrong > 0:
    print(f"\nClassification Report ({best_name}):")
    print(classification_report(y_t, best_pred))

## 25 · Interpretation and Business Insight

**Key findings:**
- **Mg (Magnesium)** and **Al (Aluminum)** are the most discriminative features.
- **Ba (Barium)** distinguishes Type 7 from others.
- **Types 1 and 2** (window glass) are most common and easiest to classify.
- **Types 5, 6** (containers, tableware) are rare and harder.

**Forensic takeaway:** Glass oxide composition provides reliable type identification for criminal investigations.

## 26 · Limitations

1. Only 214 samples — high-variance estimates.
2. Type 4 missing — incomplete class coverage.
3. Only 9 features — missing other physical properties.
4. No provenance information (manufacturer, age).
5. Small test set makes metrics unreliable.

## 27 · How to Improve This Project

1. Collect more samples, especially for rare types.
2. Add features: density, hardness, thickness.
3. Use cross-validation for more robust estimates.
4. Try TabPFN-v2 (designed for small datasets).
5. Bootstrap confidence intervals on metrics.

## 28 · Production Considerations

- Portable spectrometer for field analysis.
- Calibrate with known glass standards.
- Handle unknown glass types gracefully.
- Uncertainty quantification for forensic evidence.

## 29 · Common Mistakes

1. Not encoding target to 0-based for XGBoost.
2. Overfitting on 214 rows with complex models.
3. Using accuracy alone on imbalanced 6-class problem.
4. Not stratifying the train/test split.
5. Ignoring that Type 4 is absent.

## 30 · Mini Challenge / Exercises

1. Add 5-fold cross-validation — are metrics more stable?
2. Remove Ba feature — which class suffers most?
3. Try KNN classifier — does it beat trees?
4. Plot 2D PCA projection colored by glass type.
5. Merge Types 1+2+3 vs 5+6+7 — does binary classification work better?

## 31 · Final Summary / Key Takeaways

1. **Chemical composition** reliably identifies glass type.
2. **Small datasets** (214 rows) amplify overfitting risk.
3. **Mg and Ba** are the most discriminative elements.
4. **Boosting models** work well even on small multi-class problems.
5. **Simple baselines** are essential for context on tiny datasets.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    y_t = y_test if isinstance(y_test, np.ndarray) else y_test.values
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_t, yp)), 4),
        "f1": round(float(f1_score(y_t, yp, average='weighted')), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))